### 🖼️ Dataset Thumbnails (NomaDamas___ko-vdr-train-public)

![Thumbnail](../thumbnails/NomaDamas___ko-vdr-train-public_01.png)

In [ ]:
import numpy as np
from datasets import load_dataset, get_dataset_config_names
import random
from PIL import Image
import os

# 튜터 코멘트:
# 🌟 오늘의 실습 목표: '스마트 문서 감별사(Smart Document Inspector)' AI 만들기! 🌟
# 이 데이터셋은 텍스트 질문과 관련 이미지/문서가 함께 들어있어요.
# 목표는 단순히 질문을 던지는 것을 넘어, 질문을 바탕으로 '어떤 부분이 중요하고, 어떤 이미지를 봐야 할지'를 AI가 분석하는 과정을 코드로 구현해보는 거예요!

# 데이터셋 정보
DATASET_NAME = "NomaDamas/ko-vdr-train-public"
SPLIT_NAME = "train"
SAMPLE_COUNT = 10 # 초보자 실습이므로, 딱 10개의 샘플만 사용합니다!

print("✨ [🤖 AI 튜터] 환영합니다! 오늘은 멀티모달(텍스트+이미지) 데이터셋을 다뤄볼 거예요. 화이팅! 💪")
print("="*60)

# --- 1. 데이터셋 로드 전략 (스트리밍 vs 일반 모드) ---

# 튜터 코멘트: 데이터셋이 너무 크기 때문에, 무조건 '스트리밍' 방식으로 로드하는 것이 중요해요!
# 하지만 스트리밍 방식이 때때로 까다로울 수 있어서, 예외 처리를 통해 안전하게 로드할 거예요.

print(f"[📖 1/4] 📚 데이터셋 로딩을 시작합니다: {DATASET_NAME}...")

dataset = None
try:
    # 1차 시도: 스트리밍(Streaming) 방식으로 로드 (가장 빠르고 메모리 효율적)
    dataset = load_dataset(DATASET_NAME, split=SPLIT_NAME, streaming=True)
    print("✅ 성공! 스트리밍 모드(Streaming)로 데이터셋을 성공적으로 로드했습니다. (메모리 절약!)")
except Exception as e:
    # 2차 시도: 스트리밍이 안 되거나 문제가 발생할 경우, 제한적으로 다운로드
    print(f"⚠️ 스트리밍 로드 실패 ({e}). 소량의 데이터만 일반 모드로 로드합니다.")
    try:
        dataset = load_dataset(DATASET_NAME, split=SPLIT_NAME)
    except Exception as e2:
        print(f"🛑 치명적인 오류 발생: 데이터셋을 로드할 수 없습니다. 오류: {e2}")
        # 여기까지만 실행하고 스크립트를 중단하는 것이 안전합니다.
        raise SystemExit("데이터셋 로드 실패로 인해 스크립트를 종료합니다.")

# --- 2. 샘플 데이터 추출 및 준비 ---

# 튜터 코멘트: 데이터셋 전체를 돌리는 건 너무 오래 걸리니까, '맛보기'로 10개만 뽑아와서 실습할게요!
# 스트리밍이든 일반이든, .take(K) 메서드를 사용해서 상위 K개만 가져오는 것이 핵심입니다!

if hasattr(dataset, "take"):
    # .take()가 존재하면 스트리밍 데이터셋(IterableDataset)인 경우
    print(f"\n[✨ 2/4] ✨ {SAMPLE_COUNT}개의 샘플을 준비합니다...")
    sampled_dataset_iterator = dataset.take(SAMPLE_COUNT)
else:
    # 일반 데이터셋 (Dataset)인 경우
    print(f"\n[✨ 2/4] ✨ {SAMPLE_COUNT}개의 샘플을 준비합니다...")
    sampled_dataset_iterator = dataset.take(SAMPLE_COUNT)

# 성능 최적화를 위해 리스트로 변환하여 반복합니다.
sample_data_list = list(sampled_dataset_iterator)

if not sample_data_list:
    print("😭 준비할 샘플이 없습니다. 데이터셋 로드에 실패했는지 확인해주세요.")
    exit()

print(f"✅ 준비 완료! {len(sample_data_list)}개의 샘플을 가지고 실습을 시작합니다.")
print("="*60)


# --- 3. 실습 메인 루프: '스마트 문서 감별사' 역할 수행하기 ---

print("\n\n=== 🧠 [실습 시작] 스마트 문서 감별사 AI 작동 시뮬레이션 ===")

for i, sample in enumerate(sample_data_list):
    print(f"\n=== 📌 [샘플 {i+1}/{len(sample_data_list)}] === 🔍")

    # 1. 필수 데이터 추출 (친절한 변수명 사용)
    query = sample.get('query', '쿼리가 없습니다.')
    markdown_context = sample.get('markdown', '문서 컨텍스트가 없습니다.')
    image = sample.get('image')
    doc_id = sample.get('doc_id', 'N/A')

    print(f"🔍 [Task] 문서 ID: {doc_id} | 질문: '{query[:30]}...'")

    # 2. AI 분석 단계 1: 질문과 문서 컨텍스트 분석 (Text Analysis)
    print("  [🤖 AI 분석 1/3] 📑 텍스트 흐름 분석:")
    
    # 튜터 코멘트: 질의응답(Q&A) 태스크의 핵심은 '핵심 키워드' 찾기예요.
    # 여기서 간단한 가짜 키워드 추출 로직을 구현하여 AI가 질문에서 중요한 단어를 뽑아내는 과정을 시뮬레이션해봅니다.
    
    keywords = []
    temp_query = query.lower()
    if '무엇' in temp_query or '어디' in temp_query:
        keywords.append("핵심어 추출 필요 (Wh-Question)")
    elif "보고서" in temp_query or "문서" in temp_query:
        keywords.append("보고서, 문서 (Document Context)")
    else:
        keywords.append("일반 키워드 추출 (Keywords)")

    print(f"    ➡️ 추출된 중요 키워드 추론: {', '.join(keywords)}")
    
    # 3. AI 분석 단계 2: 이미지 데이터 검사 (Visual Analysis)
    print("  [🤖 AI 분석 2/3] 🖼️ 시각 자료 분석 (VDR):")
    
    if image is None:
        print("    ➡️ 이미지가 없습니다. 텍스트 정보만으로 판단합니다.")
    else:
        # 튜터 코멘트: 이미지 객체가 들어와있으면, 우리는 이 이미지가 질문과 *관련*한지 검토해야 합니다.
        try:
            # PIL Image 객체로 존재 유무 및 기본 정보를 확인 (실제 CNN 모델 필요)
            img_size = np.array(image).shape
            print(f"    ✅ 이미지 발견! 크기: {img_size[0]}x{img_size[1]} 픽셀.")
            print("    💡 (실제 AI 과정) 이미지 임베딩을 통해 질문과 시각적 관련성(Relevance)을 계산할 예정입니다.")
        except Exception as e:
            print(f"    ❌ 이미지 처리 오류 발생: {e}. (이미지 데이터의 형식이 다를 수 있습니다.)")


    # 4. AI 분석 단계 3: 최종 판단 및 추천 (Decision & Recommendation)
    print("  [🤖 AI 분석 3/3] 🎯 최종 결론 도출:")
    
    if keywords and "보고서" in keywords[-1] and image is not None:
        print(f"    ✨ **결론:** 이 질문은 '{query}'에 대한 **[문서 내용을 근거로 한]** 시각적 확인이 필요합니다.")
        print("    ✅ 추천 행동: [Markdown Context]를 1차 검색하여 문서를 특정하고, [Image]를 확대하여 시각적 증거를 찾아야 합니다.")
    elif keywords and "핵심어 추출 필요" in keywords[-1]:
         print(f"    ✨ **결론:** 핵심 질문입니다. 문서 전체를 탐색하기보다, '{keywords[0]}'에 관련된 페이지를 먼저 필터링하는 것이 효율적입니다.")
    else:
        print("    💡 **결론:** 기본적인 정보 탐색 질문으로 보입니다. 문서 컨텍스트를 바탕으로 답변을 생성하겠습니다.")
    
    print("-" * 60)


print("\n\n✨ [🎉 실습 완료] 축하합니다! ✨")
print("위 코드를 통해 멀티모달 AI가 어떻게 텍스트와 이미지를 종합적으로 분석하는지 시뮬레이션해봤어요.")
print("가장 어려웠던 부분은, 이미지 분석(CNN)과 텍스트 분석(NLP)을 결합하는 '융합' 과정이라는 것을 기억해주세요!")